# Setup dependencies

Before you begin, you will need your API Token from AI Hub.
To get this value, log into AI Hub and copy it from here: https://app.aihub.qualcomm.com/account/
Then open the "aihub_api_token.txt" file and paste it there.  Use CTRl-S to save the file and close it.

In [ ]:
%%time
import sys

![ ! -s "aihub_api_token.txt" ] && echo "Error: place your AI Hub token in aihub_api_token.txt file!"

!sudo apt-get install -y -q libgl1
!{sys.executable} -m pip uninstall -y -q opencv-python
!{sys.executable} -m pip install -q opencv-python-headless
!{sys.executable} -m pip install -q qai_hub

# Use a local copy of ultralytics to patch round() error issue
!{sys.executable} -m pip uninstall -q -y ultralytics
!git config --global user.email "you@example.com"
!git config --global user.name "Your Name"
![ ! -d "ultralytics" ] && git clone https://github.com/ultralytics/ultralytics -b v8.3.31
!cd ultralytics/ && {sys.executable} -m pip install -q .

![ -s "aihub_api_token.txt" ] && qai-hub configure --api_token $(cat aihub_api_token.txt)

# Configure settings

In [ ]:
# Limit the dataset to 5 classes in order to test training quickly, change to False for full training
SAMPLE_ONLY = True
DATASET_NAME = "CUB_200_2011"
DATASET_FILENAME = DATASET_NAME + ".tgz"
LABELS_FILENAME = DATASET_NAME + ".labels"
LABELS_COLOR = "0x00FF00FF"
DATA_DIR = DATASET_NAME + "/"

# Download and prepare the dataset

In [ ]:
%%time
import os
import pandas as pd
import shutil

from PIL import Image

def convert_coco_to_yolo(img_size, bbox):
    x_center = (2*bbox[0] + bbox[2])/(2*img_size[0])
    y_center = (2*bbox[1] + bbox[3])/(2*img_size[1])
    width = bbox[2]/img_size[0]
    height = bbox[3]/img_size[1]
    return (round(x_center, 6), round(y_center, 6), round(width, 6), round(height, 6))

def append_file(filename, line):
    with open(filename, "a") as file:
        file.write(line)
        file.close()

![ ! -f "$DATASET_FILENAME" ] && [ ! -d "$DATA_DIR" ] && wget --no-check-certificate -q -O $DATASET_FILENAME https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz?download=1

# Unzip and cleanup the old compressed file
![ ! -d "$DATA_DIR" ] && tar -xf $DATASET_FILENAME

# Remove data archive
!rm -rf $DATASET_FILENAME

if SAMPLE_ONLY:
    CLASS_FILTER = [17, 36, 47, 68, 73]
    TRAINING_EPOCHS = 100
else:
    CLASS_FILTER = []
    TRAINING_EPOCHS = 1200

# read main list of images
df = pd.read_csv(DATA_DIR + "images.txt", sep=' ',
                 names=["id", "filepath"])
# merge list of image_id to class labels
df = df.merge(pd.read_csv(DATA_DIR + "image_class_labels.txt", sep=' ',
                          names=["id", "class_id"]), on="id")
# merge list of image_id to training flag
df = df.merge(pd.read_csv(DATA_DIR + "train_test_split.txt", sep=' ',
                          names=["id", "training"]), on="id")
# merge list of image_id to bounding box data
df = df.merge(pd.read_csv(DATA_DIR + "bounding_boxes.txt", sep=' ',
                          names=["id", "x_min", "y_min", "width", "height"]), on="id")

classes_df = pd.read_csv(DATA_DIR + "classes.txt", sep=' ',
                         names=["class_id", "class_name"])
df = df.merge(classes_df, on="class_id")

# Create dataset folders
!rm -rf datasets/
!mkdir -p datasets/$DATASET_NAME/images/train
!mkdir -p datasets/$DATASET_NAME/images/val
!mkdir -p datasets/$DATASET_NAME/images/test
!mkdir -p datasets/$DATASET_NAME/labels/train
!mkdir -p datasets/$DATASET_NAME/labels/val

# Remove old labels file
!rm -rf $LABELS_FILENAME

# copy the dataset template
!cp CUB_200_2011.yaml.template datasets/CUB_200_2011.yaml

class_ids = sorted(df['class_id'].drop_duplicates())

class_counter = -1
for c_id in class_ids:
    if len(CLASS_FILTER) == 0 or c_id in CLASS_FILTER:
        class_counter += 1

        # Parse the class name
        c_name = classes_df[classes_df['class_id'] == c_id]['class_name'].array[0].split(".")[1]
        print(f"Parsing: {c_name}")
        # append to the dataset config
        append_file(f"datasets/{DATASET_NAME}.yaml", f"  {class_counter}: {c_name}\n")
        # append to the labels file
        append_file(LABELS_FILENAME, f'(structure)"{c_name.replace(" ", "-").replace("_", "-").lower()},id=(guint)0x{class_counter:0>4X},color=(guint){LABELS_COLOR};"\n')

        for image_id in df[df['class_id'] == c_id]['id']:
            image_dfs = df[df['id'] == image_id]
            filepath_orig = image_dfs['filepath'].array[0]
            filename_new = filepath_orig.split("/")[1]
            label_filename = filename_new.split(".")[0] + ".txt"

            # convert from bounding box data to YOLO style x_center,y_center,w,h box
            img_size = Image.open(f"{DATASET_NAME}/images/{filepath_orig}").size
            bbox = (image_dfs['x_min'].array[0], image_dfs['y_min'].array[0], image_dfs['width'].array[0], image_dfs['height'].array[0])
            yolo_box = convert_coco_to_yolo(img_size, bbox)

            if image_dfs['training'].array[0] == 1:
                loc = "train"
            if image_dfs['training'].array[0] == 0:
                loc = "val"

            # TODO: copy -> rename
            shutil.copy(f"{DATASET_NAME}/images/{filepath_orig}", f"datasets/{DATASET_NAME}/images/{loc}/{filename_new}")
            # Create label file: <class_id> <x_center> <y_center> <width> <height>
            with open(f"datasets/{DATASET_NAME}/labels/{loc}/{label_filename}", "w") as label_file:
                label_file.write(f"{class_counter} {yolo_box[0]} {yolo_box[1]} {yolo_box[2]} {yolo_box[3]}\n")

# Download test images
!wget --no-check-certificate -q -O datasets/$DATASET_NAME/images/test/multi-goldfinch-1.jpg https://t3.ftcdn.net/jpg/01/44/64/36/500_F_144643697_GJRUBtGc55KYSMpyg1Kucb9yJzvMQooW.jpg
!wget --no-check-certificate -q -O datasets/$DATASET_NAME/images/test/northern-flicker-1.jpg https://upload.wikimedia.org/wikipedia/commons/5/5c/Northern_Flicker_%28Red-shafted%29.jpg
!wget --no-check-certificate -q -O datasets/$DATASET_NAME/images/test/northern-cardinal-1.jpg https://cdn.pixabay.com/photo/2013/03/19/04/42/bird-94957_960_720.jpg
!wget --no-check-certificate -q -O datasets/$DATASET_NAME/images/test/blue-jay-1.jpg https://cdn12.picryl.com/photo/2016/12/31/blue-jay-bird-feather-animals-b8ee04-1024.jpg
!wget --no-check-certificate -q -O datasets/$DATASET_NAME/images/test/hummingbird-1.jpg http://res.freestockphotos.biz/pictures/17/17875-hummingbird-close-up-pv.jpg

# Remove the original data now that we've created the dataset
!rm -rf $DATA_DIR

# Create model

In [ ]:
# Download a pre-trained model with weights: YOLOv5 (N)
![ ! -f "yolov5nu.pt" ] && wget --no-check-certificate -q https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov5nu.pt

# Train and Validate

In [ ]:
from ultralytics import YOLO

# Load our model
train_model = YOLO("yolov5nu.pt")

# Train the model on the CUB-200-2011 dataset for 100 epochs (limited training)
results = train_model.train(data=f"datasets/{DATASET_NAME}.yaml", epochs=100)

# Backup best.pt after training
!cp runs/detect/train/weights/best.pt yolov5nu_train.pt

# Run inference on test images

In [ ]:
%%time

# Clean up previous tests
!mkdir -p tests

import os

from ultralytics import YOLO

# Load our model
test_model = YOLO("yolov5nu_train.pt")

# Run inference with the YOLOv5n model on the test images
directory = f"datasets/{DATASET_NAME}/images/test/"
for filename in os.listdir(directory):
    f = os.path.join(directory, filename)
    results = test_model(f)
    for r in results:
        r.save(filename=f"tests/{filename}")  # save to disk

# Quantize on AI Hub
Warning: You needed to set your API Token at the top of this notebook before proceeding.

In [ ]:
%%time

# Based on AI Hub docs: https://app.aihub.qualcomm.com/docs/hub/quantize_examples.html

import os
import numpy as np
import torch
import torchvision

#from PIL import Image
from ultralytics import YOLO

import qai_hub as hub

# 1. Load our model and export as ONNX
torch_model = YOLO("yolov5nu_train.pt")

# 2. Trace the model to TorchScript format
input_shape = (1, 3, 640, 640)
pt_model = torch.jit.trace(torch_model, torch.rand(input_shape))

# 3. Compile the model to ONNX
device = hub.Device("RB3 Gen 2 (Proxy)")
compile_onnx_job = hub.submit_compile_job(
    model=pt_model,
    device=device,
    input_specs=dict(image_tensor=input_shape),
    options="--target_runtime onnx",
)
assert isinstance(compile_onnx_job, hub.CompileJob)

unquantized_onnx_model = compile_onnx_job.get_target_model()
assert isinstance(unquantized_onnx_model, hub.Model)

# 4. Load and pre-process downloaded calibration data
sample_inputs = []

images_dir = f"datasets/{DATASET_DIR}/images/val/"
for image_path in os.listdir(images_dir):
    image = Image.open(os.path.join(images_dir, image_path))
    image = image.convert("RGB").resize(input_shape[2:])
    sample_input = np.array(image).astype(np.float32) / 255.0
    sample_input = np.expand_dims(np.transpose(sample_input, (2, 0, 1)), 0)
    sample_inputs.append(sample_input)
calibration_data = dict(image_tensor=sample_inputs)

# 5. Quantize the model
quantize_job = hub.submit_quantize_job(
    model=unquantized_onnx_model,
    calibration_data=calibration_data,
    weights_dtype=hub.QuantizeDtype.INT8,
    activations_dtype=hub.QuantizeDtype.INT8,
)

quantized_onnx_model = quantize_job.get_target_model()
assert isinstance(quantized_onnx_model, hub.Model)